# Practical 2 — Superstore order size

R Jupyter version of the course material. Select an **R** kernel in Positron and run cells from top to bottom, starting with the setup cell. Keep the complete course folder, data and images together.

Generated from the matching QMD; make teaching edits in the QMD and regenerate. The published HTML/PDF retains the full course styling.


In [ ]:
# Notebook sessions may start in the notebook folder.
# Find the opened course folder so the original data/... paths work unchanged.
.course_root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(.course_root, "_quarto.yml")) ||
       !dir.exists(file.path(.course_root, "data"))) {
  .course_parent <- dirname(.course_root)
  if (identical(.course_parent, .course_root)) {
    stop("Open the complete course folder containing _quarto.yml and data/.")
  }
  .course_root <- .course_parent
}
setwd(.course_root)
rm(.course_root)
if (exists(".course_parent", inherits = FALSE)) rm(.course_parent)


In [ ]:
#| label: numerical-import
source("lessons/_includes/course-tools.R")
lines <- read.csv("data/business/superstore-lines.csv",
                  stringsAsFactors = FALSE, colClasses = c(postal_code = "character"))
orders <- make_orders(lines)
stopifnot(nrow(orders) == n_distinct(lines$order_id),
          isTRUE(all.equal(sum(orders$sales), sum(lines$sales))))
knitr::kable(head(orders[c("order_id", "sales", "profit", "quantity", "line_count")] ), digits = 2)


- `data/business/superstore-lines.csv`; guide: `data/business/README.md`.
- Sales, profit and quantity are **summed** across an order’s lines.
- Do not average line sales to estimate average order sales.
- The reconciliation checks that aggregation neither loses nor duplicates revenue.

## 3. Look at the distribution first


In [ ]:
#| fig-alt: "Histogram of all order sales, retaining the long right tail."
ggplot(orders, aes(sales)) +
  geom_histogram(bins = 35, fill = petra_ink, colour = "white") +
  labs(x = "Order sales (USD)", y = "Orders")


- Most orders are relatively small; a few are very large.
- The long right tail affects the mean and SD.
- A histogram’s bin width changes the visible detail.
- A tall bar shows many orders in an interval, not many dollars of revenue.


In [ ]:
#| fig-alt: "The order-sales histogram with a labelled logarithmic sales axis."
ggplot(orders, aes(sales)) +
  geom_histogram(bins = 25, fill = petra_ink, colour = "white") +
  scale_x_log10() + labs(x = "Order sales ($; log10 axis)", y = "Orders")


- The log axis makes the lower range easier to inspect without deleting large orders.
- Equal spacing on this axis means equal ratios, not equal dollar differences.

## 4. Choose useful numerical measures


In [ ]:
x <- orders$sales
summary_table <- data.frame(
  statistic = c("Orders", "Mean (USD)", "Median (USD)", "Sample SD (USD)",
                "Minimum (USD)", "Q1 (USD)", "Q3 (USD)", "Maximum (USD)", "IQR (USD)"),
  value = c(length(x), mean(x), median(x), sd(x), min(x),
            quantile(x, .25, type = 7), quantile(x, .75, type = 7), max(x), IQR(x)))
knitr::kable(summary_table, digits = 2)


- Mean order sales: about **\$471**.
- Median order sales: about **\$154**.
- Mean explains revenue per order; median describes the middle order.
- SD is sensitive to the right tail; IQR describes the middle half.
- `sd()` uses denominator $`n-1`$; the frame SD uses $`N`$ and is slightly smaller here.

### Revenue reconciliation


In [ ]:
c(total_sales = sum(x), mean_times_orders = mean(x) * length(x),
  median_times_orders = median(x) * length(x))


- Mean × number of orders reproduces total revenue.
- Median × number of orders generally does not.
- The median is not an alternative way to calculate the total.

### Absolute and relative spread


In [ ]:
c(range_dollars = diff(range(x)),
  mean_absolute_deviation = mean(abs(x - mean(x))),
  coefficient_of_variation_pct = 100 * sd(x) / mean(x))


- Mean absolute deviation above is not R’s `mad()`, which uses the median and a scaling constant.
- CV is appropriate here because sales have a meaningful zero and positive mean.
- Do not apply the same CV rule to profits that can be negative or near zero on average.

## 5. Percentiles for operational planning

> What order values define the upper 10% and upper 5% of the observed distribution?


In [ ]:
knitr::kable(data.frame(percentile = c("50th", "75th", "90th", "95th", "99th"),
  order_sales = as.numeric(quantile(x, c(.5, .75, .9, .95, .99), type = 7))), digits = 2)


- Percentiles describe positions in the observed distribution.
- Interpolation and ties mean a threshold may not leave exactly the named percentage strictly above it.
- A sales percentile does not directly measure parcel size, picking time or staffing needs.
- Additional operational variables would be needed for those decisions.

## 6. Boxplots and unusual orders


In [ ]:
#| fig-alt: "Boxplot of all order sales; points beyond the whiskers identify unusual records to investigate."
ggplot(orders, aes(x = "All orders", y = sales)) +
  geom_boxplot(fill = petra_accent, outlier.alpha = .3) +
  labs(x = NULL, y = "Order sales (USD)")

quartiles <- quantile(x, c(.25, .75), type = 7)
lower_fence <- quartiles[1] - 1.5 * IQR(x)
upper_fence <- quartiles[2] + 1.5 * IQR(x)
flagged <- orders %>% filter(sales < lower_fence | sales > upper_fence)
c(lower_fence = lower_fence, upper_fence = upper_fence, flagged_orders = nrow(flagged))


- Flags identify records worth investigating.
- The source contains genuine large recorded transactions; a flag alone does not establish an error.
- Boxplot hinges and reported quartiles can differ slightly for some sample sizes.


In [ ]:
knitr::kable(orders %>% arrange(desc(sales)) %>%
  select(order_id, sales, profit, line_count, quantity) %>% head(5), digits = 2)


- Inspect the underlying product lines before deciding what a large order means.
- A large order may contain one costly product or several products.
- Keep all records in the main summary.

### A labelled sensitivity comparison


In [ ]:
central_orders <- orders %>% filter(sales >= lower_fence, sales <= upper_fence)
knitr::kable(data.frame(
  scope = c("Full frame — primary result", "Within IQR fences — sensitivity only"),
  n = c(nrow(orders), nrow(central_orders)),
  mean_sales = c(mean(orders$sales), mean(central_orders$sales)),
  median_sales = c(median(orders$sales), median(central_orders$sales))), digits = 2)


- The second row changes the scope; it is not a corrected estimate of the original frame mean.
- Large changes show why valid extreme values matter to the business.
- Do not hide exclusions or present the smaller mean as the retailer’s full average order value.

## 7. Independent operations briefing

1.  Compare order sales across regions using $`n`$, mean, median, SD and IQR.
2.  Draw side-by-side boxplots, with the same scale for all regions.
3.  Find the region with the largest **total revenue** and the largest **median order**. Are they the same?
4.  Identify the top five orders by sales and inspect their product lines.
5.  Calculate the share of total revenue contributed by the top 10% of orders ranked by sales. State how you handle ties or rounding of the number of orders.
6.  Write a short briefing distinguishing typical size, variability and revenue concentration.


**Manual setup — run once in the R Console if needed.** This cell is excluded from Run All.

```r
#| eval: false
# group_by(region) %>% summarise(...) builds the regional table.
# arrange(desc(sales)) ranks order records.
# An explicit rule is to take ceiling(0.10 * nrow(orders)) highest-ranked orders.
```


**Your conclusion should include:**

- The scope and order-level denominator.
- Two numerical findings with dollar units.
- One useful operational follow-up.
- A limitation: sales values alone do not measure physical workload.

## 8. Continue

- [Practical 3: Discounts and profitability](https://msfasha.github.io/307312-Applied-Statistics/extended-cases/superstore/practical-03-association-and-groups.html).
- `data/business/README.md`.
